# 02 - Предобработка данных

In [19]:
import json          # Для чтения/записи JSON-файлов (метрики, конфиги)
import sys
from pathlib import Path  # Удобная работа с путями к файлам

import pandas as pd  # Таблицы

sys.path.append("../src")  # Добавляем папку src, чтобы найти utils.py
import utils

SEED = 42                          # Фиксируем случайность для воспроизводимости
RAW_PATH = "../data/raw/diabetic_data.csv"           # Путь к сырым данным
PROCESSED_DIR = Path("../data/processed")            # Куда сохраним обработанные данные
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)     # Создаём папку, если её нет

## 1. Загрузка данных

In [20]:
# Загружаем данные. utils.load_data заменяет "?" на NaN автоматически.
df = utils.load_data(RAW_PATH)
print(f"Загружено: {df.shape[0]:,} строк, {df.shape[1]} признаков")
df.head(3)

Загружено: 101,766 строк, 50 признаков


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## 2. Дедупликация

Из анализа следует, что в датасете один пациент может встречаться несколько раз, потому что у него могло быть несколько госпитализаций.

Чтобы избежать такой ситуации, оставляем только первый визит каждого пациента. Для этого данные сортируются по `encounter_id`, а затем удаляются повторные строки по `patient_nbr`. После этого каждый `patient_nbr` должен встречаться только один раз.

In [21]:
# Шаг 1: Сортируем по encounter_id (номер визита).
# Меньший encounter_id = более ранний визит = первый визит пациента.
# Шаг 2: Удаляем дубликаты по patient_nbr, оставляем только ПЕРВЫЙ визит (keep='first').
# Зачем? Если оставить все визиты — один пациент может попасть и в train, и в test.
# Тогда модель "видела" его при обучении. Это data leakage: результаты будут завышены.
df = df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")

# Проверяем: теперь каждый patient_nbr должен встречаться ровно один раз
assert df["patient_nbr"].is_unique, "patient_nbr не уникален после дедупликации!"
print(f"После дедупликации: {df.shape[0]:,} строк (уникальных пациентов)")

После дедупликации: 71,518 строк (уникальных пациентов)


## 3. Удалить умерших и с хосписом

Для задачи повторной госпитализации эти строки лучше исключить, потому что у таких пациентов уже нет обычного сценария возвращения в больницу в течение 30 дней.

In [22]:
# Коды discharge_disposition_id соответствующие смерти или выписке в хоспис.
# Источник: официальный справочник (IDS_mapping.csv).
# 11 = Умер в стационаре
# 13 = Хоспис — домашний уход
# 14 = Хоспис — медучреждение
# 19, 20, 21 = Другие смертельные исходы
DEAD_HOSPICE = {11, 13, 14, 19, 20, 21}

before = df.shape[0]
# Удаляем строки с этими кодами — такие пациенты физически не могут быть реадмитированы.
# Их "NO" в readmitted правильное, но по другой причине, чем у обычных пациентов.
# Оставлять их = добавлять систематический шум в негативный класс.
df = df[~df["discharge_disposition_id"].isin(DEAD_HOSPICE)].copy()
print(f"Удалено умерших/хоспис: {before - df.shape[0]:,} строк. Осталось: {df.shape[0]:,}")

Удалено умерших/хоспис: 1,545 строк. Осталось: 69,973


## 4. Целевая переменная

В датасете `readmitted` имеет три значения: `<30`, `>30` и `NO`. Нас интересует именно ранняя повторная госпитализация, поэтому значение `<30` переводим в `1`, а все остальные случаи в `0`. После создания `target` исходный столбец `readmitted` удаляется.

In [23]:
# Создаём бинарную целевую переменную:
# 1 = пациент повторно госпитализирован в течение 30 дней ("<30")
# 0 = все остальные случаи (">30" или "NO")
df["target"] = (df["readmitted"] == "<30").astype(int)

# ВАЖНО: Удаляем исходный столбец readmitted.
# Если оставить — модель может прочитать из него ответ напрямую. Это называется target leakage.
df = df.drop(columns=["readmitted"])

pos_rate = df["target"].mean()
print(f"Баланс классов: {pos_rate:.3%} позитивных (readmitted <30)")
df["target"].value_counts()

Баланс классов: 8.971% позитивных (readmitted <30)


target
0    63696
1     6277
Name: count, dtype: int64

## 5. Удаление идентификаторов

На этом шаге удаляем столбцы `encounter_id` и `patient_nbr`.

In [24]:
# Удаляем идентификаторы.
# encounter_id — уникальный номер визита (строки). Не несёт медицинского смысла.
# patient_nbr — ID пациента. Тоже не несёт смысла и уже не нужен после дедупликации.
# Если оставить — модель может переобучиться на конкретных пациентах.
df = df.drop(columns=["encounter_id", "patient_nbr"])
print("Количество колонок после удаления идентификаторов:", df.shape[1])

Количество колонок после удаления идентификаторов: 48


## 6. Обработка пропусков

Сразу удаляем `weight`, `max_glu_serum` и `A1Cresult`, потому что в них слишком много пропущенных значений. Признак `payer_code` тоже часто отсутствует и не является основным медицинским признаком для задачи повторной госпитализации. Признаки `medical_specialty` и `race` не удаляем сразу. Для них пропущенные значения заменяются на категорию `Unknown`.

Также удаляем строки, где `gender` равен `Unknown/Invalid`, потому что таких записей очень мало.

Основные пропуски должны остаться только в диагнозах `diag_1`, `diag_2` и `diag_3`, которые будут обработаны на следующем шаге

In [25]:
# Обработка пропущенных значений.

# Колонки с критически большим числом пропусков — удаляем полностью.
# weight: ~97% пропусков — почти нет данных, колонка бесполезна.
# max_glu_serum, A1Cresult: >50% пропусков, к тому же медицински они дублируют другие признаки.
df = df.drop(columns=["weight", "max_glu_serum", "A1Cresult"])

# payer_code: ~40% пропусков + не является медицинским признаком — удаляем.
df = df.drop(columns=["payer_code"])

# medical_specialty (~49% пропусков) и race (~2% пропусков) — не удаляем.
# Заменяем NaN на категорию 'Unknown': отсутствие данных само по себе может быть информативным.
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
df["race"] = df["race"].fillna("Unknown")

# Удаляем 3 строки с невалидным полом — их слишком мало для отдельной категории.
before = df.shape[0]
df = df[df["gender"] != "Unknown/Invalid"].copy()
print(f"Удалено невалидных gender: {before - df.shape[0]}. Осталось: {df.shape[0]:,}")

print("Оставшиеся NaN по колонкам (>0):")
missing = df.isnull().sum()
print(missing[missing > 0])

Удалено невалидных gender: 3. Осталось: 69,970
Оставшиеся NaN по колонкам (>0):
diag_1      10
diag_2     293
diag_3    1224
dtype: int64


## 7. Группировка диагнозов (ICD-9 → 9 категорий)

В признаках `diag_1`, `diag_2` и `diag_3` хранятся коды диагнозов ICD-9. На этапе анализа было видно, что у этих признаков сотни уникальных значений. Поэтому ICD-9 коды переводятся в укрупненные медицинские группы по стандартным диапазонам. Например, отдельно выделяются заболевания системы кровообращения, дыхательной системы, пищеварительной системы и тд.

Диабет выделяется отдельно, потому что это центральное заболевание в данном датасете. Коды, которые начинаются с `250`, переводятся в категорию `Diabetes`.

Пропущенные, пустые и нераспознанные значения относятся к категории `Other`. Также в `Other` отправляются E-коды и V-коды, потому что они не являются обычными числовыми диагнозами: E-коды описывают внешние причины, а V-коды - дополнительные факторы обращения за медицинской помощью.

In [26]:
def map_icd9(code) -> str:
    """Переводим ICD-9 код диагноза в укрупнённую клиническую категорию.

    Зачем: в diag_1/2/3 хранится 700+ уникальных кодов вида '250.1', '428', 'E895'.
    Такую кардинальность (количество уникальных значений) нельзя скормить модели напрямую:
    получатся тысячи редких признаков. Группируем по стандартным диапазонам ICD-9.
    """
    # Если значение пустое или знак вопроса — это "неизвестный диагноз"
    if pd.isna(code) or str(code).strip() in ("", "?"):
        return "Other"

    code = str(code).strip()

    # Диабет выделен отдельно — это центральное заболевание в датасете
    if code.startswith("250"):
        return "Diabetes"

    # E-коды: внешние причины (травмы, отравления), V-коды: доп. факторы обращения
    # Они не являются стандартными числовыми диагнозами
    if code.startswith(("E", "V")):
        return "Other"

    try:
        num = float(code)  # Пробуем преобразовать в число
    except ValueError:
        return "Other"  # Не получилось — нестандартный код

    # Диапазоны ICD-9 по клиническим системам организма:
    if 1 <= num <= 139:
        return "Infectious"       # Инфекционные болезни
    if 140 <= num <= 239:
        return "Neoplasms"        # Новообразования (опухоли)
    if 240 <= num <= 279:
        return "Endocrine"        # Эндокринные (диабет уже перехвачен выше)
    if 280 <= num <= 289:
        return "Blood"            # Болезни крови
    if 290 <= num <= 319:
        return "Mental"           # Психические расстройства
    if 320 <= num <= 389:
        return "Nervous"          # Нервная система
    if 390 <= num <= 459:
        return "Circulatory"      # Сердечно-сосудистая система
    if 460 <= num <= 519:
        return "Respiratory"      # Дыхательная система
    if 520 <= num <= 579:
        return "Digestive"        # Пищеварительная система
    if 580 <= num <= 629:
        return "Genitourinary"    # Мочеполовая система
    if 630 <= num <= 679:
        return "Pregnancy"        # Беременность и роды
    if 680 <= num <= 709:
        return "Skin"             # Кожа
    if 710 <= num <= 739:
        return "Musculoskeletal"  # Костно-мышечная система
    if 740 <= num <= 759:
        return "Congenital"       # Врождённые аномалии
    if 760 <= num <= 779:
        return "Perinatal"        # Перинатальные состояния
    if 780 <= num <= 799:
        return "Symptoms"         # Симптомы и неточно обозначенные состояния
    if 800 <= num <= 999:
        return "Injury"           # Травмы и отравления

    return "Other"


# Применяем функцию к каждому диагнозу в трёх столбцах
for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].map(map_icd9)

print("diag_1 distribution:")
print(df["diag_1"].value_counts())
print(df["diag_2"].value_counts())
print(df["diag_3"].value_counts())

diag_1 distribution:
diag_1
Circulatory        21315
Respiratory         6446
Digestive           6325
Diabetes            5748
Symptoms            5503
Injury              4692
Musculoskeletal     4064
Genitourinary       3414
Neoplasms           2538
Endocrine           1850
Skin                1780
Infectious          1685
Mental              1545
Other                929
Nervous              858
Blood                651
Pregnancy            586
Congenital            41
Name: count, dtype: int64
diag_2
Circulatory        21781
Diabetes            9700
Respiratory         6445
Endocrine           5605
Genitourinary       5042
Symptoms            3168
Digestive           2704
Skin                2228
Other               2080
Blood               2075
Mental              1856
Injury              1822
Neoplasms           1599
Musculoskeletal     1295
Infectious          1240
Nervous              894
Pregnancy            353
Congenital            83
Name: count, dtype: int64
diag_3
Circul

## 8. Feature engineering и SHAP-отбор признаков

На этом шаге формируется итоговый набор признаков для дальнейшего обучения моделей.

Сначала `admission_type_id`, `discharge_disposition_id` и `admission_source_id` переводятся в строковый тип. Хотя эти признаки записаны числами, по смыслу это категориальные коды из справочника, а не обычные количественные значения.

Затем добавляются новые признаки: общее количество прошлых обращений пациента, флаги наличия прошлых обращений и стационарных госпитализаций, среднее число процедур в день, отношение количества лекарств к лабораторным процедурам и середина возрастного интервала.

Финальный список признаков берется из отдельного ноутбука `shap.ipynb`. Там RandomForest используется для SHAP-ранжирования признаков, а разные top-N наборы проверяются через 3-fold CV на Logistic Regression. По F2 лучшим оказался вариант top-25, поэтому здесь фиксируется этот набор признаков.

In [27]:
# admission/discharge/source ID — это категориальные коды из справочника, а не числа.
# Если оставить как int — модель посчитает, что ID=10 "больше" ID=2, что бессмысленно.
for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    df[col] = df[col].astype("Int64").astype("string")

# Защита от деления на ноль: минимум 1 день в больнице
time = df["time_in_hospital"].clip(lower=1)

# --- Новые признаки (feature engineering) ---
# Суммарное число предыдущих обращений пациента (до текущей госпитализации)
df["total_prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]

# Флаг: был ли пациент когда-либо госпитализирован раньше (0/1)
df["has_prior_visits"] = (df["total_prior_visits"] > 0).astype(int)

# Флаг: была ли хоть одна стационарная госпитализация в прошлом
df["has_inpatient"] = (df["number_inpatient"] > 0).astype(int)

# Среднее число процедур в день (интенсивность лечения)
df["procedures_per_day"] = df["num_procedures"] / time

# Отношение лекарств к лабораторным процедурам (насколько "лекарственное" лечение)
df["med_lab_ratio"] = df["num_medications"] / (df["num_lab_procedures"] + 1)

# Числовая середина возрастного диапазона (например, "[50-60)" → 55)
# Нужна, потому что возраст хранится как текстовый диапазон, а не число
age_mid_map = {
    "[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35, "[40-50)": 45,
    "[50-60)": 55, "[60-70)": 65, "[70-80)": 75, "[80-90)": 85, "[90-100)": 95,
}
df["age_mid"] = df["age"].map(age_mid_map).astype(float)

# Загружаем финальный список признаков из notebooks/shap/shap.ipynb.
# Как работал SHAP-отбор:
#   1. Собрали ~37 кандидатов: числовые признаки + категориальные + ~23 признака лекарств.
#   2. RandomForest обучен на подвыборке (12 000 строк), посчитаны SHAP-значения.
#   3. Признаки ранжированы по средней |SHAP|. One-hot колонки лекарств схлопнуты обратно в один признак.
#   4. Проверены варианты top-N (10, 12, 15, 20, 25, 30, all) через 3-fold CV на Logistic Regression.
#   5. Лучший по F2 оказался top-25 (ROC-AUC CV = 0.644).
# Итог: 25 признаков — 12 числовых + 13 категориальных (включая лекарства: insulin, metformin, glipizide).
selection_report_path = Path("../results/feature_selection/shap_selection_report.json")
with open(selection_report_path, encoding="utf-8") as f:
    selection_report = json.load(f)

numeric_features = selection_report["numeric"]       # 12 числовых признаков
categorical_features = selection_report["categorical"]  # 13 категориальных признаков
selected_features = selection_report["selected_features"]  # Итоговый список 25 признаков

# Оставляем только выбранные признаки + целевую переменную
df = df[selected_features + ["target"]].copy()

# Сохраняем отчёт об отборе признаков для документации
data_engineering_report = {
    "method": selection_report["method"],
    "encoding_checked": ["onehot"],
    "best_cv_variant": f"shap_top_{selection_report['best_top_n']}_logreg_onehot",
    "cv_evidence": {
        "best_top_n": selection_report["best_top_n"],
        "logreg_3fold_roc_auc_mean": selection_report["logreg_3fold_roc_auc_mean"],
        "logreg_3fold_f2_mean": selection_report["logreg_3fold_f2_mean"],
    },
    "selected_features": selected_features,
}

print(f"Числовых признаков: {len(numeric_features)}")
print(f"Категориальных признаков: {len(categorical_features)}")
print("Выбранные признаки:", selected_features)


Числовых признаков: 12
Категориальных признаков: 13
Выбранные признаки: ['discharge_disposition_id', 'number_inpatient', 'has_inpatient', 'time_in_hospital', 'procedures_per_day', 'age', 'age_mid', 'diag_3', 'has_prior_visits', 'diag_1', 'diag_2', 'medical_specialty', 'number_diagnoses', 'insulin', 'num_lab_procedures', 'total_prior_visits', 'diabetesMed', 'med_lab_ratio', 'num_medications', 'admission_source_id', 'admission_type_id', 'metformin', 'number_outpatient', 'glipizide', 'change']


## 9. Привести категориальные к dtype category

На этом шаге все признаки из списка `categorical_features` переводятся в тип `category`.

In [28]:
# Переводим категориальные признаки в тип pandas 'category'.
# Это не просто экономия памяти — многие модели (CatBoost, TabM) явно ожидают category-dtype,
# чтобы правильно обрабатывать категории (кодировать, обрабатывать неизвестные значения).
for col in categorical_features:
    df[col] = df[col].astype("category")

# Смотрим итоговое распределение типов данных в таблице
print(df.dtypes.astype(str).value_counts())

category    13
int64       10
float64      3
Name: count, dtype: int64


## 10. Stratified-сплит 60/20/20

На этом шаге готовый датасет делится на три части: обучающую, валидационную и тестовую выборки.

Используется стратифицированное разбиение по `target`. Это важно, потому что положительный класс редкий: пациентов с повторной госпитализацией в течение 30 дней намного меньше, чем остальных. Стратификация сохраняет примерно одинаковую долю положительного класса во всех трех выборках.

In [29]:
# Делим данные на 3 части: train (обучение), val (подбор гиперпараметров), test (финальная оценка).
# Пропорция 60/20/20 — стандартная для таких задач.
#
# СТРАТИФИКАЦИЯ по target обязательна при дисбалансе классов.
# Без стратификации случайное разбиение может поместить почти все позитивные примеры в train,
# а в val/test почти не останется пациентов с реадмиссией — оценка будет нерепрезентативной.
train, val, test = utils.split(df, target_col="target", random_state=SEED)

# Проверяем баланс классов в каждом сплите
for name, split_df in [("train", train), ("val", val), ("test", test)]:
    rate = split_df["target"].mean()
    print(f"{name:5s}: {len(split_df):,} строк | pos rate = {rate:.3%}")
# Ожидаем ~11% позитивных в каждом сплите — стратификация работает.

train: 41,982 строк | pos rate = 8.971%
val  : 13,994 строк | pos rate = 8.975%
test : 13,994 строк | pos rate = 8.968%


## 11. Сохранение результатов

In [30]:
# Сохраняем три сплита в формате Parquet.
# Parquet лучше CSV: хранит типы данных (category, Int64), занимает меньше места, загружается быстрее.
train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
val.to_parquet(PROCESSED_DIR / "val.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

# Сохраняем списки числовых и категориальных признаков.
# Это нужно последующим ноутбукам (catboost, tabm и т.д.):
# каждая модель должна знать, какие признаки числовые (нормализуются),
# а какие категориальные (кодируются или обрабатываются нативно).
feature_types = {"numeric": numeric_features, "categorical": categorical_features}
with open(PROCESSED_DIR / "feature_types.json", "w", encoding="utf-8") as f:
    json.dump(feature_types, f, indent=2, ensure_ascii=False)
with open(PROCESSED_DIR / "data_engineering_report.json", "w", encoding="utf-8") as f:
    json.dump(data_engineering_report, f, indent=2, ensure_ascii=False)

print("Сохранено в", PROCESSED_DIR)
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {p.name:30s}  {p.stat().st_size / 1024:.1f} KB")

Сохранено в ..\data\processed
  data_engineering_report.json    0.9 KB
  feature_types.json              0.6 KB
  test.parquet                    199.5 KB
  train.parquet                   546.5 KB
  val.parquet                     200.4 KB
